### Test di Inferenza con Qwen3-ASR + LoRA

In [ ]:
import torch
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration
from peft import PeftModel
from datasets import load_dataset, Audio

print("Caricamento dei modelli in corso...")

# 1. Percorsi
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"
ADAPTER_PATH = "./qwen3_asr_lora_waxal/checkpoint-100"

# 2. Carica il Modello Base e il Processor
processor = AutoProcessor.from_pretrained(MODEL_ID)
base_model = Qwen3ASRForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 3. Applica i pesi LoRA appena addestrati
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Modello pronto!")

In [ ]:
# 4. Prendi un file di test (il primo del dataset streaming)
ds = load_dataset("google/WaxalNLP", name="lug_asr", split="train", streaming=True)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
sample = next(iter(ds))

print("\n--- TEST INFERENZA ---")
print("TESTO ORIGINALE REALE :", sample["transcription"])

# 5. Prepara l'input per Qwen3 (usando lo stesso template del training)
conversation = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "dummy"}, 
        {"type": "text", "text": "Trascrivi questo audio."}
    ]}
]

text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
inputs = processor(text=text, audio=sample["audio"]["array"], sampling_rate=16000, return_tensors="pt")

# Sposta i tensori sulla scheda video
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# 6. Genera la trascrizione
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

# Qwen3 restituisce l'input concatenato all'output, quindi tagliamo via la parte dell'input
generated_ids = generated_ids[:, inputs['input_ids'].shape[1]:]
transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("TESTO PREDITTO DA NOI :", transcription)